In [ ]:
"""
env setup:
conda create -y --name tf1_model_export python=3.7
conda activate tf1_model_export
# note: gpu support is not necessary for tensorflow
pip install "tensorflow<2"
pip install "csbdeep[tf1]"
# also install stardist in this example
pip install "stardist[tf1]"
"""

In [ ]:
from __future__ import print_function, unicode_literals, absolute_import, division
import sys
import numpy as np
import matplotlib
matplotlib.rcParams["image.interpolation"] = 'none'
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

from glob import glob
from tqdm import tqdm
from tifffile import imread
from csbdeep.utils import Path, normalize

from stardist import fill_label_holes, random_label_cmap, calculate_extents, gputools_available
from stardist.matching import matching, matching_dataset
from stardist.models import Config2D, StarDist2D, StarDistData2D
from csbdeep.utils import normalize
from stardist.plot import render_label
from skimage.color import rgb2hed

np.random.seed(42)
lbl_cmap = random_label_cmap()

In [ ]:
# prints a list of available starDist models
StarDist2D.from_pretrained()

In [ ]:
# select a model
model = StarDist2D.from_pretrained('Versatile (H&E nuclei)')

In [ ]:
X = sorted(glob('training_data/images/*.tif'))
Y = sorted(glob('training_data/masks/*.tif'))
assert all(Path(x).name==Path(y).name for x,y in zip(X,Y))

In [ ]:
# NOTE: StarDist requires images to be at least the patch size (e.g. 512x512).
# If height or width is smaller than 512, training will fail with:
# "ValueError: Some images are too small for given patch_size".

In [ ]:
bad_files = []

for x_path, y_path in zip(X, Y):
    x = imread(x_path)
    if x.shape[0] < 512 or x.shape[1] < 512:
        bad_files.append((x_path, x.shape))

for path, shape in bad_files:
    print(path, shape)

In [ ]:
print(len(X), len(Y))

In [ ]:
X = list(map(imread,X))
Y = list(map(imread,Y))

n_channel = 1 if X[0].ndim == 2 else X[0].shape[-1]
print(f"Number of channels = {n_channel} ")

In [ ]:
axis_norm = (0,1)   # normalize channels independently
# axis_norm = (0,1,2) # normalize channels jointly

X = [normalize(x,1,99,axis=axis_norm) for x in tqdm(X)] # same parameters as in qupath script
Y = [fill_label_holes(y) for y in tqdm(Y)]

In [ ]:
# splitting data into training and validation
assert len(X) > 1, "not enough training data"
rng = np.random.RandomState(42)
ind = rng.permutation(len(X))
n_val = max(1, int(round(0.15 * len(ind))))
ind_train, ind_val = ind[:-n_val], ind[-n_val:]
X_val, Y_val = [X[i] for i in ind_val]  , [Y[i] for i in ind_val]
X_trn, Y_trn = [X[i] for i in ind_train], [Y[i] for i in ind_train] 
print('number of images: %3d' % len(X))
print('- training:       %3d' % len(X_trn))
print('- validation:     %3d' % len(X_val))

In [ ]:
# plot train image and its mask
def plot_img_label(img, lbl, img_title="image", lbl_title="label", **kwargs):
    fig, (ai,al) = plt.subplots(1,2, figsize=(12,5), gridspec_kw=dict(width_ratios=(1.25,1)))
    im = ai.imshow(img, cmap='gray', clim=(0,1))
    ai.set_title(img_title)    
    fig.colorbar(im, ax=ai)
    al.imshow(lbl, cmap=lbl_cmap)
    al.set_title(lbl_title)
    plt.tight_layout()

In [ ]:
i = min(60, len(X)-1)
img, lbl = X[i], Y[i]
assert img.ndim in (2,3)
img = img if (img.ndim==2 or img.shape[-1]==3) else img[...,0]
plot_img_label(img,lbl)
None;

In [ ]:
print(model.config) # model configuration

In [ ]:
# Data augmentation

def random_fliprot(img, mask): 
    assert img.ndim >= mask.ndim
    axes = tuple(range(mask.ndim))
    perm = tuple(np.random.permutation(axes))
    img = img.transpose(perm + tuple(range(mask.ndim, img.ndim))) 
    mask = mask.transpose(perm) 
    for ax in axes: 
        if np.random.rand() > 0.5:
            img = np.flip(img, axis=ax)
            mask = np.flip(mask, axis=ax)
    return img, mask 

def random_intensity_change(img):
    img = img*np.random.uniform(0.6,2) + np.random.uniform(-0.2,0.2)
    return img


def augmenter(x, y):
    """Augmentation of a single input/label image pair.
    x is an input image
    y is the corresponding ground-truth label image
    """
    x, y = random_fliprot(x, y)
    x = random_intensity_change(x)
    # add some gaussian noise
    sig = 0.02*np.random.uniform(0,1)
    x = x + sig*np.random.normal(0,1,x.shape)
    return x, y

In [ ]:
# plot some augmented examples
img, lbl = X[60],Y[60]
plot_img_label(img, lbl)
for _ in range(3):
    img_aug, lbl_aug = augmenter(img,lbl)
    plot_img_label(img_aug, lbl_aug, img_title="image augmented", lbl_title="label augmented")

In [ ]:
# Training
history = model.train(
    X_trn, Y_trn,
    validation_data=(X_val, Y_val),
    augmenter=augmenter,
    epochs=50,
    steps_per_epoch=30
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Train loss')
plt.plot(history.history['val_loss'], label='Val loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('StarDist Training Curve')
plt.legend()
plt.grid()
plt.show()

In [ ]:
model.optimize_thresholds(X_val, Y_val)

In [ ]:
# Predict the labels for all validation images:
Y_val_pred = [model.predict_instances(x, n_tiles=model._guess_n_tiles(x), show_tile_progress=False)[0]
              for x in tqdm(X_val)]

In [ ]:
plot_img_label(X_val[11],Y_val[11], lbl_title="label GT")
plot_img_label(X_val[11],Y_val_pred[11], lbl_title="label Pred")

In [ ]:
idx = 11

img = X_val[idx]
pred_labels = Y_val_pred[idx]  
gt_labels = Y_val[idx]  

In [ ]:
gt = gt_labels
pred = pred_labels

# normalize to binary if needed
gt_bin = (gt > 0).astype(np.uint8)
pred_bin = (pred > 0).astype(np.uint8)

overlay = np.zeros((*gt.shape, 3), dtype=np.float32)

# Red = Ground Truth
overlay[..., 0] = gt_bin

# Green = Prediction
overlay[..., 1] = pred_bin

# Calculate overlap (yellow regions)
intersection = np.logical_and(gt_bin, pred_bin).sum()
union = np.logical_or(gt_bin, pred_bin).sum()

overlap_percent = (intersection / union * 100) if union > 0 else 0

# Plot
plt.figure(figsize=(6,6))
plt.imshow(overlay)

plt.title('')

# Legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor='red', label='Ground Truth (GT)'),
    Patch(facecolor='lightgreen', label='Prediction'),
    Patch(facecolor='yellow', label='Overlap (GT ∩ Prediction)')
]

plt.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.05),
    ncol=3
)
plt.axis("off")
plt.show()

In [ ]:
model.export_TF('he_retrained')
from zipfile import ZipFile
with ZipFile('he_retrained.zip', 'r') as zipObj:
   zipObj.extractall('c:/temp')
import os
os.system('python -m tf2onnx.convert --opset 10 --saved-model "c:/temp" --output_frozen_graph "./he_retrained.pb"')

In [ ]:
import shutil

src = "NEW_2D_versatile_he_retrained.pb"
dst = "/mnt/c/Users/19utk/Downloads/starDist_models/NEW_2D_versatile_he_retrained.pb"

shutil.copy(src, dst)